# Phase 0: Data Cleaning - Building Clean Master Timeline from Raw Data

## Objective
Load raw $LogFile and $UsnJrnl CSV files, apply labels from suspicious files, and create a clean master timeline with proper timestamp features.

## Critical Issues Being Fixed
1. **Broken anomaly features** - Will calculate from actual MAC timestamps (not eventtime)
2. **Missing values** - Proper handling of LogFile-only vs UsnJrnl-only records
3. **Duplicates** - Remove exact duplicate events
4. **Label conflicts** - Resolve LogFile vs UsnJrnl disagreements

## Input
- **LogFile CSVs:** `data/raw/logfile/` (12 cases)
- **UsnJrnl CSVs:** `data/raw/usnjrnl/` (12 cases)
- **Labels:** `data/raw/suspicious/` (12 cases)

## Output
- **Cleaned Timeline:** `data/processed/Phase 0 - Data Cleaning/master_timeline_cleaned.csv`
- **Quality Report:** `data/processed/Phase 0 - Data Cleaning/data_quality_report.txt`

---
## 1. Setup & Imports

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print("Python executable:", sys.executable)


✓ Libraries imported successfully
Pandas version: 2.3.2
NumPy version: 2.3.3
Python executable: /Users/soni/Github/Digital-Detectives_Thesis/.venv/bin/python


In [14]:
# Define paths - resolve to absolute paths
BASE_DIR = Path.cwd() 
RAW_DIR = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 0 - Data Cleaning'

LOGFILE_DIR = RAW_DIR / 'logfile'
USNJRNL_DIR = RAW_DIR / 'usnjrnl'
SUSPICIOUS_DIR = RAW_DIR / 'suspicious'

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths exist
print("📂 Directory Configuration:")
print(f"  Base Dir:   {BASE_DIR}")  # ← Shows absolute path
print(f"  LogFile:    {LOGFILE_DIR} {'✓' if LOGFILE_DIR.exists() else '✗'}")
print(f"  UsnJrnl:    {USNJRNL_DIR} {'✓' if USNJRNL_DIR.exists() else '✗'}")
print(f"  Suspicious: {SUSPICIOUS_DIR} {'✓' if SUSPICIOUS_DIR.exists() else '✗'}")
print(f"  Output:     {OUTPUT_DIR} ✓")

📂 Directory Configuration:
  Base Dir:   /Users/soni/Github/Digital-Detectives_Thesis
  LogFile:    /Users/soni/Github/Digital-Detectives_Thesis/data/raw/logfile ✓
  UsnJrnl:    /Users/soni/Github/Digital-Detectives_Thesis/data/raw/usnjrnl ✓
  Suspicious: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/suspicious ✓
  Output:     /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 0 - Data Cleaning ✓


---
## 2. Load LogFile Data (12 Cases)

In [15]:
print("=" * 80)
print("LOADING LOGFILE DATA")
print("=" * 80)

logfile_dfs = []

for case_id in range(1, 13):
    filename = f"{case_id:02d}-PE-LogFile.csv"
    filepath = LOGFILE_DIR / filename
    
    if not filepath.exists():
        print(f"⚠️  Case {case_id:02d}: File not found - {filename}")
        continue
    
    try:
        df = pd.read_csv(filepath, encoding='utf-8-sig', low_memory=False)
        df['case_id'] = case_id
        df['source'] = 'logfile'
        logfile_dfs.append(df)
        print(f"✓ Case {case_id:02d}: Loaded {len(df):,} records from {filename}")
    except Exception as e:
        print(f"✗ Case {case_id:02d}: Error loading {filename} - {e}")

# Combine all LogFile data
logfile_df = pd.concat(logfile_dfs, ignore_index=True)

print(f"\n✅ Total LogFile records: {len(logfile_df):,}")
print(f"   Cases loaded: {logfile_df['case_id'].nunique()}")
print(f"   Columns: {list(logfile_df.columns)}")

LOADING LOGFILE DATA
✓ Case 01: Loaded 39,077 records from 01-PE-LogFile.csv
✓ Case 02: Loaded 14,783 records from 02-PE-LogFile.csv
✓ Case 03: Loaded 24,063 records from 03-PE-LogFile.csv
✓ Case 04: Loaded 12,731 records from 04-PE-LogFile.csv
✓ Case 05: Loaded 14,242 records from 05-PE-LogFile.csv
✓ Case 06: Loaded 14,030 records from 06-PE-LogFile.csv
✓ Case 07: Loaded 23,737 records from 07-PE-LogFile.csv
✓ Case 08: Loaded 23,379 records from 08-PE-LogFile.csv
✓ Case 09: Loaded 25,688 records from 09-PE-LogFile.csv
✓ Case 10: Loaded 23,932 records from 10-PE-LogFile.csv
✓ Case 11: Loaded 14,083 records from 11-PE-LogFile.csv
✓ Case 12: Loaded 14,139 records from 12-PE-LogFile.csv

✅ Total LogFile records: 243,884
   Cases loaded: 12
   Columns: ['LSN', 'EventTime(UTC+8)', 'Event', 'Detail', 'File/Directory Name', 'Full Path', 'CreationTime', 'ModifiedTime', 'MFTModifiedTime', 'AccessedTime', 'Redo', 'Target VCN', 'Cluster Index', 'case_id', 'source']


---
## 3. Load UsnJrnl Data (12 Cases)

In [16]:
print("=" * 80)
print("LOADING USNJRNL DATA")
print("=" * 80)

usnjrnl_dfs = []

for case_id in range(1, 13):
    filename = f"{case_id:02d}-PE-UsnJrnl.csv"
    filepath = USNJRNL_DIR / filename
    
    if not filepath.exists():
        print(f"⚠️  Case {case_id:02d}: File not found - {filename}")
        continue
    
    try:
        df = pd.read_csv(filepath, encoding='utf-8-sig', low_memory=False)
        df['case_id'] = case_id
        df['source'] = 'usnjrnl'
        usnjrnl_dfs.append(df)
        print(f"✓ Case {case_id:02d}: Loaded {len(df):,} records from {filename}")
    except Exception as e:
        print(f"✗ Case {case_id:02d}: Error loading {filename} - {e}")

# Combine all UsnJrnl data
usnjrnl_df = pd.concat(usnjrnl_dfs, ignore_index=True)

print(f"\n✅ Total UsnJrnl records: {len(usnjrnl_df):,}")
print(f"   Cases loaded: {usnjrnl_df['case_id'].nunique()}")
print(f"   Columns: {list(usnjrnl_df.columns)}")

LOADING USNJRNL DATA
✓ Case 01: Loaded 316,817 records from 01-PE-UsnJrnl.csv
✓ Case 02: Loaded 247,386 records from 02-PE-UsnJrnl.csv
✓ Case 03: Loaded 245,425 records from 03-PE-UsnJrnl.csv
✓ Case 04: Loaded 263,451 records from 04-PE-UsnJrnl.csv
✓ Case 05: Loaded 265,287 records from 05-PE-UsnJrnl.csv
✓ Case 06: Loaded 264,518 records from 06-PE-UsnJrnl.csv
✓ Case 07: Loaded 247,908 records from 07-PE-UsnJrnl.csv
✓ Case 08: Loaded 248,604 records from 08-PE-UsnJrnl.csv
✓ Case 09: Loaded 249,559 records from 09-PE-UsnJrnl.csv
✓ Case 10: Loaded 249,438 records from 10-PE-UsnJrnl.csv
✓ Case 11: Loaded 264,432 records from 11-PE-UsnJrnl.csv
✓ Case 12: Loaded 265,621 records from 12-PE-UsnJrnl.csv

✅ Total UsnJrnl records: 3,128,446
   Cases loaded: 12
   Columns: ['TimeStamp(UTC+8)', 'USN', 'File/Directory Name', 'FullPath', 'EventInfo', 'SourceInfo', 'FileAttribute', 'Carving Flag', 'FileReferenceNumber', 'ParentFileReferenceNumber', 'case_id', 'source']


---
## 4. Load Suspicious Labels (Ground Truth)

In [17]:
print("=" * 80)
print("LOADING SUSPICIOUS LABELS")
print("=" * 80)

suspicious_dfs = []

for case_id in range(1, 13):
    filename = f"{case_id:02d}-PE-Suspicious.csv"
    filepath = SUSPICIOUS_DIR / filename
    
    if not filepath.exists():
        print(f"⚠️  Case {case_id:02d}: File not found - {filename}")
        continue
    
    try:
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        df['case_id'] = case_id
        suspicious_dfs.append(df)
        
        # Count labels by category
        timestomp_count = df[df['category'] == 'Timestamp Manipulation'].shape[0]
        tool_exec_count = df[df['category'] == 'Execution of Suspicious Programs'].shape[0]
        
        print(f"✓ Case {case_id:02d}: {len(df)} labels (Timestomp: {timestomp_count}, Tool Exec: {tool_exec_count})")
    except Exception as e:
        print(f"✗ Case {case_id:02d}: Error loading {filename} - {e}")

# Combine all suspicious labels
suspicious_df = pd.concat(suspicious_dfs, ignore_index=True)

print(f"\n✅ Total suspicious events: {len(suspicious_df):,}")
print(f"   Cases loaded: {suspicious_df['case_id'].nunique()}")
print(f"\n📊 Label Distribution:")
print(suspicious_df.groupby(['source', 'category']).size())

LOADING SUSPICIOUS LABELS
✓ Case 01: 4 labels (Timestomp: 2, Tool Exec: 2)
✓ Case 02: 3 labels (Timestomp: 1, Tool Exec: 2)
✓ Case 03: 4 labels (Timestomp: 2, Tool Exec: 2)
✓ Case 04: 58 labels (Timestomp: 58, Tool Exec: 0)
✓ Case 05: 1 labels (Timestomp: 1, Tool Exec: 0)
✓ Case 06: 72 labels (Timestomp: 72, Tool Exec: 0)
✓ Case 07: 4 labels (Timestomp: 2, Tool Exec: 2)
✓ Case 08: 35 labels (Timestomp: 33, Tool Exec: 2)
✓ Case 09: 38 labels (Timestomp: 36, Tool Exec: 2)
✓ Case 10: 31 labels (Timestomp: 31, Tool Exec: 0)
✓ Case 11: 93 labels (Timestomp: 91, Tool Exec: 2)
✓ Case 12: 161 labels (Timestomp: 159, Tool Exec: 2)

✅ Total suspicious events: 504
   Cases loaded: 12

📊 Label Distribution:
source   category                        
logfile  Execution of Suspicious Programs      8
         Timestamp Manipulation               14
usnjrnl  Execution of Suspicious Programs      8
         Timestamp Manipulation              474
dtype: int64


---
## 5. Standardize Column Names

LogFile and UsnJrnl have different column names. We need to standardize them for merging.

In [18]:
print("=" * 80)
print("STANDARDIZING COLUMN NAMES")
print("=" * 80)

# LogFile column mapping
logfile_rename = {
    'LSN': 'lf_lsn',
    'EventTime(UTC+8)': 'eventtime',
    'Event': 'lf_event',
    'Detail': 'lf_detail',
    'File/Directory Name': 'filename',
    'Full Path': 'filepath',
    'CreationTime': 'lf_creation_time',
    'ModifiedTime': 'lf_modified_time',
    'MFTModifiedTime': 'lf_mft_modified_time',
    'AccessedTime': 'lf_accessed_time',
    'Redo': 'lf_redo',
    'Target VCN': 'lf_target_vcn',
    'Cluster Index': 'lf_cluster_index'
}

# UsnJrnl column mapping
usnjrnl_rename = {
    'TimeStamp(UTC+8)': 'eventtime',
    'USN': 'usn_usn',
    'File/Directory Name': 'filename',
    'FullPath': 'filepath',
    'EventInfo': 'usn_event_info',
    'SourceInfo': 'usn_source_info',
    'FileAttribute': 'usn_file_attribute',
    'Carving Flag': 'usn_carving_flag',
    'FileReferenceNumber': 'usn_file_reference_number',
    'ParentFileReferenceNumber': 'usn_parent_file_reference_number'
}

# Apply renames
logfile_df = logfile_df.rename(columns=logfile_rename)
usnjrnl_df = usnjrnl_df.rename(columns=usnjrnl_rename)

print("✓ Standardized LogFile columns")
print("✓ Standardized UsnJrnl columns")
print(f"\n📋 Common columns: eventtime, filename, filepath, case_id, source")

STANDARDIZING COLUMN NAMES
✓ Standardized LogFile columns
✓ Standardized UsnJrnl columns

📋 Common columns: eventtime, filename, filepath, case_id, source


---
## 6. Create Master Timeline

Combine LogFile and UsnJrnl into a single timeline.

In [19]:
print("=" * 80)
print("CREATING MASTER TIMELINE")
print("=" * 80)

# Get all unique columns
all_columns = set(logfile_df.columns) | set(usnjrnl_df.columns)

# Add missing columns to each dataframe
for col in all_columns:
    if col not in logfile_df.columns:
        logfile_df[col] = np.nan
    if col not in usnjrnl_df.columns:
        usnjrnl_df[col] = np.nan

# Combine
master_timeline = pd.concat([logfile_df, usnjrnl_df], ignore_index=True)

# Sort by case_id and eventtime
master_timeline['eventtime'] = pd.to_datetime(master_timeline['eventtime'], errors='coerce')
master_timeline = master_timeline.sort_values(['case_id', 'eventtime'], ignore_index=True)

print(f"\n✅ Master timeline created:")
print(f"   Total records: {len(master_timeline):,}")
print(f"   LogFile records: {(master_timeline['source'] == 'logfile').sum():,}")
print(f"   UsnJrnl records: {(master_timeline['source'] == 'usnjrnl').sum():,}")
print(f"   Missing eventtime: {master_timeline['eventtime'].isnull().sum():,}")

CREATING MASTER TIMELINE

✅ Master timeline created:
   Total records: 3,372,330
   LogFile records: 243,884
   UsnJrnl records: 3,128,446
   Missing eventtime: 147,991


---
## 7. Save Master Timeline (Initial Version)

Save the merged timeline before applying labels and cleaning.

In [20]:
print("=" * 80)
print("SAVING MASTER TIMELINE")
print("=" * 80)

# Define output file path
output_file = OUTPUT_DIR / 'master_timeline_raw.csv'

print(f"\n💾 Saving to: {output_file}")
print(f"   Records: {len(master_timeline):,}")
print(f"   Columns: {len(master_timeline.columns)}")

# Save to CSV
master_timeline.to_csv(output_file, index=False, encoding='utf-8-sig')

# Verify file was created
if output_file.exists():
    file_size_mb = output_file.stat().st_size / (1024 * 1024)
    print(f"\n✅ File saved successfully!")
    print(f"   Size: {file_size_mb:.2f} MB")
    print(f"   Location: {output_file}")
else:
    print(f"\n❌ Error: File was not created!")

# Also save suspicious labels for reference
suspicious_output = OUTPUT_DIR / 'suspicious_labels.csv'
suspicious_df.to_csv(suspicious_output, index=False, encoding='utf-8-sig')
print(f"\n✅ Suspicious labels saved: {suspicious_output}")

print(f"\n📊 Summary:")
print(f"   Master timeline: {len(master_timeline):,} events")
print(f"   Suspicious labels: {len(suspicious_df):,} labels")
print(f"   Output directory: {OUTPUT_DIR}")

SAVING MASTER TIMELINE

💾 Saving to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 0 - Data Cleaning/master_timeline_raw.csv
   Records: 3,372,330
   Columns: 22

✅ File saved successfully!
   Size: 851.23 MB
   Location: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 0 - Data Cleaning/master_timeline_raw.csv

✅ Suspicious labels saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 0 - Data Cleaning/suspicious_labels.csv

📊 Summary:
   Master timeline: 3,372,330 events
   Suspicious labels: 504 labels
   Output directory: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 0 - Data Cleaning


---
## ✅ Phase 0 - Part 1 Complete!

### What We Accomplished:
1. ✅ Loaded 12 LogFile CSVs (243,884 records)
2. ✅ Loaded 12 UsnJrnl CSVs (3,128,446 records)
3. ✅ Loaded 504 suspicious labels (488 timestomp events, 16 tool executions)
4. ✅ Standardized column names across both sources
5. ✅ Created unified master timeline (3,372,330 total events)
6. ✅ Saved raw timeline (851 MB) and labels to output directory

### Current Data State:
- **Total Events:** 3,372,330
- **Missing Eventtime:** 147,991 (4.4%)
- **Timestomping Labels:** 
  - LogFile: 14 events
  - UsnJrnl: 474 events
  - **Total: 488 actual timestomping events**
- **Tool Execution Labels:** 16 events (features, not labels)

---

## 📋 Next Steps (To Be Implemented):

### Part 2: Apply Labels & Create Timestamp Features
**Notebook:** `02_Label_Application_and_Timestamps.ipynb`

**Tasks:**
1. **Apply Labels from Suspicious CSVs**
   - Match suspicious events to master timeline by LSN/USN
   - Create `is_timestomped` column (binary label)
   - Create `timestomp_tool_executed` column (feature, not label)
   - Handle label conflicts (when LF and USN disagree)

2. **Create Proper MAC Timestamp Features**
   - Parse `lf_creation_time`, `lf_modified_time`, `lf_accessed_time`
   - Calculate timestamp anomalies:
     - `creation_after_modification` (seconds gap)
     - `accessed_before_creation` (seconds gap)
     - `mac_all_identical` (binary)
     - `mac_time_range` (max - min in seconds)
     - `future_timestamp_days` (how many days in future)
   - **These will replace the broken features that had 0.0 importance!**

3. **Handle Missing Values**
   - Remove 147,991 events with missing `eventtime`
   - Strategy for LogFile-only vs UsnJrnl-only records
   - Expected: ~3.2M events remaining

4. **Remove Duplicates**
   - Identify exact duplicate events
   - Keep first occurrence
   - Expected: Some reduction from 3.2M

5. **Save Cleaned Timeline**
   - Output: `master_timeline_cleaned.csv`
   - Include all new timestamp features
   - Include labels applied

---

### Part 3: Data Quality Report
**Notebook:** `03_Data_Quality_Report.ipynb`

**Generate Report:**
- Label distribution by case
- Timestamp anomaly statistics
- Missing value analysis
- Duplicate analysis
- Final dataset summary

---

## 🎯 Expected Outcomes After Phase 0:

### Clean Dataset:
- **Records:** ~3.1-3.2M (after removing missing/duplicates)
- **Timestomped Events:** 488 (0.015% - extreme imbalance)
- **Proper Timestamp Features:** 
  - `creation_modification_gap` (continuous, in seconds)
  - `accessed_creation_gap` (continuous, in seconds)
  - `mac_identical` (binary)
  - `mac_time_variance` (continuous)
  - `future_days` (continuous)
  - These will have HIGH importance (not 0.0 like before!)

### Files Created:
```
data/processed/Phase 0 - Data Cleaning/
├── master_timeline_raw.csv          (851 MB) ✓ DONE
├── suspicious_labels.csv            (50 KB)  ✓ DONE
├── master_timeline_cleaned.csv      (~800 MB) ⏳ NEXT
└── data_quality_report.txt          (~10 KB)  ⏳ NEXT
```

---

## 🚀 To Continue:

**Option 1 (Recommended):** Create separate notebooks for each part
- Better organization
- Easier debugging
- Can rerun specific parts

**Option 2:** Continue in this notebook
- Add more cells below
- Implement label application
- Implement timestamp features

**Estimated Time for Part 2:** 2-3 hours of development

---

**Status:** ✅ Part 1 of Phase 0 complete - Ready for label application and timestamp feature creation!